# Notebook 1 — Train Θ_o + Generate Forget/Retain Splits

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026 · arXiv:2604.08271v1)

**Official repo:** https://github.com/tiensinh2/CMF_Unlearning

**This notebook (Part B NB1) does:**
1. Clone the official repo at `main` (pin commit for reproducibility).
2. Train Θ_o on the FULL training set using paper-faithful hyperparameters (Table 4):  
   SGD, lr=0.01, weight_decay=5e-4, momentum=0.9, CosineAnnealingLR with 5-epoch linear warmup, patience=50.
3. Generate forget/retain split files for BOTH 3:7 and 1:9 stratified random-mix ratios,  
   seeds 0/1/2, saved as `forget_indices_ratio{30|10}_seed{seed}.json`.
4. Save `theta_o_seed{seed}.pt` per seed with self-describing metadata.
5. **Evaluate Θ_o with all 3 paper metrics** (Table 1 — Original row):
   - **Output accuracy** — full forward pass retain/forget split on test set
   - **Linear Probe** — freeze encoder, train fresh linear head (50 epochs, lr=0.01)
   - **NCC** — freeze encoder, compute per-class means from train, classify test by nearest-class-center
6. Export `cmf_benchmark_config.json` consumed by NB2/3/4/5.

**Evaluation (3 metrics matching paper Table 1 — Original row):**
| Metric | Description |
|--------|-------------|
| Output | Full model forward pass — retain acc / forget acc on test set |
| Linear Probe | Freeze encoder → train nn.Linear(D,K) for 50 epochs → test retain/forget acc |
| NCC | Freeze encoder → class means from train → argmin distance on test set |

**Convention (all notebooks):** forget accuracy compared against Oracle's own forget acc  
(NOT against 0%), because under random-mix the oracle itself does not reach 0%.

> Set `TEST_MODE = True` for a ~1-min CPU smoke test. Full run: T4 GPU ~2-3 h.

## A. Environment Setup

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
# A.8 fix: clone official repo + record commit for reproducibility
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Working directory:', os.getcwd())
print('Repo commit      :', REPO_COMMIT)

In [ ]:
# ─── Config ─────────────────────────────────────────────────────────────────────────────
TEST_MODE   = False       # True → fast CPU smoke test (~1 min)
DATASET     = 'cifar10'
NUM_CLASSES = 10
ARCH        = 'resnet18'
SEEDS       = [0, 1, 2]
RATIOS      = [30, 10]    # 30 → forget 30% per class; 10 → forget 10%

# Paper Table 4 hyperparameters (CIFAR-10 ResNet-18)
# Source priority: Table 4 first, then §A.4 text for unspecified params.
# Table 4 line 2308: CIFAR-10 ResNet-18 Original lr=0.01
BATCH_SIZE    = 128
EPOCHS        = 5   if TEST_MODE else 300
PATIENCE      = 2   if TEST_MODE else 50
LR_INIT       = 1e-2   # Table 4 line 2308
WEIGHT_DECAY  = 5e-4
MOMENTUM      = 0.9
WARMUP_EPOCHS = 1   if TEST_MODE else 5
MIN_LR        = 1e-5

# Linear Probe hyperparameters — config.py EVAL dict (lp_epochs=50, lp_lr=1e-2)
LP_MAX_EPOCHS = 2   if TEST_MODE else 50   # paper: lp_epochs=50
LP_LR         = 1e-2                        # paper: lp_lr=1e-2
LP_BATCH_SIZE = 256

# NCC evaluation batch size
NCC_BATCH_SIZE = 256

CKPT_ROOT = '/kaggle/working/checkpoints/cmf_benchmark'
os.makedirs(f'{CKPT_ROOT}/pre_train',  exist_ok=True)
os.makedirs(f'{CKPT_ROOT}/splits',     exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'TEST_MODE={TEST_MODE}  EPOCHS={EPOCHS}  LP_MAX_EPOCHS={LP_MAX_EPOCHS}  device={device}')

## B. Data Loading

In [ ]:
import torchvision, torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                          download=True, transform=transform_train)
test_set   = torchvision.datasets.CIFAR10('/kaggle/working/data', train=False,
                                          download=True, transform=transform_test)

# Validation split: last 5 000 samples of train
n_val = 5000
train_idx = list(range(len(full_train) - n_val))
val_idx   = list(range(len(full_train) - n_val, len(full_train)))
train_sub = torch.utils.data.Subset(full_train, train_idx)
val_sub   = torch.utils.data.Subset(full_train, val_idx)

train_loader = torch.utils.data.DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True,
                                            num_workers=2, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_sub,   batch_size=256, shuffle=False,
                                            num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False,
                                            num_workers=2)

print(f'Train: {len(train_sub)}  Val: {len(val_sub)}  Test: {len(test_set)}')

## C. Train Θ_o — Paper-Faithful Config

In [ ]:
from models.resnet import ResNet18

def build_model():
    return ResNet18(num_classes=NUM_CLASSES, dataset=DATASET).to(device)

def make_scheduler(optimizer, warmup_epochs, total_epochs, min_lr):
    """5-epoch linear warmup → CosineAnnealingLR, as in paper §A.4."""
    warmup_sched = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                            total_iters=warmup_epochs)
    cosine_sched = CosineAnnealingLR(optimizer,
                                     T_max=total_epochs - warmup_epochs,
                                     eta_min=min_lr)
    return SequentialLR(optimizer,
                        schedulers=[warmup_sched, cosine_sched],
                        milestones=[warmup_epochs])

@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / total

def train_full(seed):
    """Train Θ_o from scratch with early stopping (patience=PATIENCE)."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = build_model()
    optimizer = optim.SGD(model.parameters(), lr=LR_INIT,
                          momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True)
    scheduler = make_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS, MIN_LR)

    best_val_acc = 0.0
    best_state   = None
    patience_cnt = 0
    early_stopped_at = None
    log = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        ep_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
        scheduler.step()

        val_acc = accuracy(model, val_loader)
        log.append({'epoch': epoch, 'val_acc': val_acc,
                    'lr': optimizer.param_groups[0]['lr']})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1

        if epoch % 10 == 0:
            print(f'  [seed{seed} epoch{epoch:3d}] val_acc={val_acc:.2f}%  '
                  f'lr={optimizer.param_groups[0]["lr"]:.2e}  '
                  f'patience={patience_cnt}/{PATIENCE}')

        if patience_cnt >= PATIENCE:
            early_stopped_at = epoch
            print(f'  Early stop at epoch {epoch} (best_val={best_val_acc:.2f}%)')
            break

    model.load_state_dict(best_state)
    test_acc = accuracy(model, test_loader)
    return model, test_acc, early_stopped_at, log

print('Helpers ready.')

In [ ]:
theta_o_models = {}
train_results  = []

for seed in SEEDS:
    tag = f'theta_o_seed{seed}'
    if TEST_MODE: tag += '_testmode'
    ckpt_path = f'{CKPT_ROOT}/pre_train/{tag}.pt'

    if os.path.exists(ckpt_path):
        print(f'[seed{seed}] Checkpoint exists, loading {ckpt_path}')
        ck = torch.load(ckpt_path, map_location=device)
        model = build_model()
        model.load_state_dict(ck['model_state_dict'])
        test_acc = accuracy(model, test_loader)
        print(f'  test_acc={test_acc:.2f}%')
    else:
        print(f'\n[seed{seed}] Training Θ_o ...')
        model, test_acc, stopped_at, log = train_full(seed)
        ck = {
            'model_state_dict': model.state_dict(),
            'config': {
                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                'lr_init': LR_INIT, 'weight_decay': WEIGHT_DECAY,
                'momentum': MOMENTUM, 'warmup_epochs': WARMUP_EPOCHS,
                'epochs': EPOCHS, 'patience': PATIENCE, 'min_lr': MIN_LR,
                'test_mode': TEST_MODE,
                'repo_commit': REPO_COMMIT,
            },
            'seed': seed,
            'metrics': {'test_acc': test_acc},
            'early_stopped_at': stopped_at,
            'train_log': log,
        }
        torch.save(ck, ckpt_path)
        print(f'  Saved {ckpt_path}  test_acc={test_acc:.2f}%')

    theta_o_models[seed] = model
    train_results.append({'seed': seed, 'test_acc': test_acc})

print('\n=== Θ_o test accuracies ===')
print(pd.DataFrame(train_results))

## D. Generate Forget/Retain Split Files

In [ ]:
# A.6 fix: ONE split protocol, generated ONCE here, loaded by ALL downstream notebooks.
# Stratified random-mix: for each class, randomly choose ratio% of its training samples
# as the forget set; the rest become the retain set.

def make_stratified_split(dataset, ratio_pct, seed):
    """Return (forget_indices, retain_indices) for a stratified random-mix split.

    Args:
        dataset  : torchvision dataset with .targets
        ratio_pct: percentage of each class to forget (e.g. 30 → 30%)
        seed     : random seed for reproducibility
    """
    rng = random.Random(seed)
    targets = np.array(dataset.targets)
    forget_idx, retain_idx = [], []
    for c in range(NUM_CLASSES):
        cls_idx = np.where(targets == c)[0].tolist()
        n_forget = max(1, int(len(cls_idx) * ratio_pct / 100))
        forget_c = rng.sample(cls_idx, n_forget)
        retain_c = [i for i in cls_idx if i not in set(forget_c)]
        forget_idx.extend(forget_c)
        retain_idx.extend(retain_c)
    return sorted(forget_idx), sorted(retain_idx)

split_info = {}
for ratio in RATIOS:
    for seed in SEEDS:
        tag = f'ratio{ratio}_seed{seed}'
        if TEST_MODE: tag += '_testmode'
        fpath = f'{CKPT_ROOT}/splits/forget_indices_{tag}.json'
        rpath = f'{CKPT_ROOT}/splits/retain_indices_{tag}.json'

        if os.path.exists(fpath):
            print(f'[{tag}] Split exists, loading.')
            with open(fpath) as f: forget_idx = json.load(f)
            with open(rpath) as f: retain_idx = json.load(f)
        else:
            forget_idx, retain_idx = make_stratified_split(full_train, ratio, seed)
            with open(fpath, 'w') as f: json.dump(forget_idx, f)
            with open(rpath, 'w') as f: json.dump(retain_idx, f)
            print(f'[{tag}] Saved: forget={len(forget_idx)}  retain={len(retain_idx)}')

        split_info[(ratio, seed)] = {'forget': forget_idx, 'retain': retain_idx}
        assert len(set(forget_idx) & set(retain_idx)) == 0, 'Overlap in splits!'
        assert len(forget_idx) + len(retain_idx) == len(full_train), 'Size mismatch!'

print('\nAll splits generated and verified.')

## E. Evaluate Θ_o on Splits

In [ ]:
# ── Section E: Evaluate Θ_o — Output / Linear Probe / NCC ────────────────────────
# Paper §3.1–§3.2, Table 1 (Original row).
# Protocol:
#   Output accuracy : test-set forward pass, retain vs forget split
#   Linear Probe    : freeze encoder, train nn.Linear for LP_MAX_EPOCHS epochs
#                     on FULL train set (D_r ∪ D_f), evaluate on test set
#   NCC             : class means from TRAIN set, argmin-distance on TEST set
#                     (paper Eq. 5 — mu_k from training data)

import copy

# ── helpers ──────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def output_retain_forget(model, test_set, forget_indices):
    """Top-1 accuracy on the retain and forget subsets of test_set."""
    train_targets  = np.array(full_train.targets)
    forget_classes = sorted(set(train_targets[forget_indices].tolist()))

    model.eval()
    all_pred, all_true = [], []
    loader = torch.utils.data.DataLoader(
        test_set, batch_size=256, shuffle=False, num_workers=2)
    for x, y in loader:
        all_pred.extend(model(x.to(device)).argmax(1).cpu().tolist())
        all_true.extend(y.tolist())

    pred   = np.array(all_pred)
    true   = np.array(all_true)
    f_mask = np.isin(true, forget_classes)
    r_mask = ~f_mask
    ret_acc = 100.0 * (pred[r_mask] == true[r_mask]).mean() if r_mask.any() else float('nan')
    for_acc = 100.0 * (pred[f_mask] == true[f_mask]).mean() if f_mask.any() else float('nan')
    return ret_acc, for_acc, forget_classes


@torch.no_grad()
def extract_features(model, loader):
    """Extract penultimate-layer (avgpool) features [N, D] from a frozen ResNet."""
    feats, labels = [], []
    buf = []

    def _hook(_m, _inp, out):
        buf.append(out.view(out.size(0), -1).detach().cpu())

    handle = model.avgpool.register_forward_hook(_hook)
    model.eval()
    for x, y in loader:
        buf.clear()
        _ = model(x.to(device))
        feats.append(buf[0])
        labels.append(y)
    handle.remove()
    return torch.cat(feats, 0).float(), torch.cat(labels, 0).long()


def linear_probe_eval(model, train_loader_full, test_set, forget_classes):
    """Train a fresh linear head on frozen features, return retain/forget test acc.

    Paper Sec 3.2: trained on full D = D_r u D_f for LP_MAX_EPOCHS epochs.
    """
    probe_model = copy.deepcopy(model).to(device)
    probe_model.eval()
    for p in probe_model.parameters():
        p.requires_grad_(False)

    Xtr, ytr = extract_features(probe_model, train_loader_full)
    test_loader_lp = torch.utils.data.DataLoader(
        test_set, batch_size=LP_BATCH_SIZE, shuffle=False, num_workers=2)
    Xte, yte = extract_features(probe_model, test_loader_lp)

    D   = Xtr.size(1)
    clf = nn.Linear(D, NUM_CLASSES).to(device)
    opt = optim.SGD(clf.parameters(), lr=LP_LR, momentum=0.9, weight_decay=0.0)
    ds  = torch.utils.data.TensorDataset(Xtr, ytr)
    dl  = torch.utils.data.DataLoader(ds, batch_size=LP_BATCH_SIZE, shuffle=True)

    for ep in range(LP_MAX_EPOCHS):
        clf.train()
        for bx, by in dl:
            opt.zero_grad()
            nn.CrossEntropyLoss()(clf(bx.to(device)), by.to(device)).backward()
            opt.step()

    clf.eval()
    with torch.no_grad():
        pred_te = clf(Xte.to(device)).argmax(1).cpu().numpy()
    true_te = yte.numpy()
    f_mask  = np.isin(true_te, forget_classes)
    r_mask  = ~f_mask
    ret_acc = 100.0 * (pred_te[r_mask] == true_te[r_mask]).mean() if r_mask.any() else float('nan')
    for_acc = 100.0 * (pred_te[f_mask] == true_te[f_mask]).mean() if f_mask.any() else float('nan')
    return ret_acc, for_acc


def ncc_eval(model, train_loader_full, test_set, forget_classes):
    """NCC: class means from train, nearest-class-center on test (paper Eq. 5).

    mu_k = mean of penultimate features for class k over the full train set.
    Assign test sample x to argmin_k ||phi(x) - mu_k||_2.
    """
    probe_model = copy.deepcopy(model).to(device)
    probe_model.eval()
    for p in probe_model.parameters():
        p.requires_grad_(False)

    # train-set features -> class means
    Xtr, ytr = extract_features(probe_model, train_loader_full)
    means = []
    for c in range(NUM_CLASSES):
        mask = (ytr == c)
        mu   = Xtr[mask].mean(0) if mask.any() else torch.zeros(Xtr.size(1))
        means.append(mu)
    M = torch.stack(means).to(device)   # [K, D]

    # test-set features -> argmin distance
    test_loader_ncc = torch.utils.data.DataLoader(
        test_set, batch_size=NCC_BATCH_SIZE, shuffle=False, num_workers=2)
    Xte, yte = extract_features(probe_model, test_loader_ncc)
    Xte = Xte.to(device)

    dists = torch.cdist(Xte.unsqueeze(0), M.unsqueeze(0)).squeeze(0)  # [N, K]
    pred  = dists.argmin(1).cpu().numpy()
    true  = yte.numpy()
    f_mask = np.isin(true, forget_classes)
    r_mask = ~f_mask
    ret_acc = 100.0 * (pred[r_mask] == true[r_mask]).mean() if r_mask.any() else float('nan')
    for_acc = 100.0 * (pred[f_mask] == true[f_mask]).mean() if f_mask.any() else float('nan')
    return ret_acc, for_acc


# ── full train loader (D_r ∪ D_f) for LP and NCC centre computation ─────────────────────
full_train_loader = torch.utils.data.DataLoader(
    full_train, batch_size=128, shuffle=False, num_workers=2
)

# ── main evaluation loop ───────────────────────────────────────────────────────────────────
eval_rows = []
for seed in SEEDS:
    model = theta_o_models[seed]
    for ratio in RATIOS:
        split          = split_info[(ratio, seed)]
        forget_idx     = split['forget']
        train_targets  = np.array(full_train.targets)
        forget_classes = sorted(set(train_targets[forget_idx].tolist()))

        print(f'\n[seed={seed} ratio={ratio}]  forget_classes={forget_classes}')

        # 1) Output accuracy
        out_ret, out_for, _ = output_retain_forget(model, test_set, forget_idx)
        print(f'  Output   retain={out_ret:.2f}%  forget={out_for:.2f}%')

        # 2) Linear Probe (paper Sec 3.2)
        lp_ret, lp_for = linear_probe_eval(model, full_train_loader, test_set, forget_classes)
        print(f'  LP       retain={lp_ret:.2f}%  forget={lp_for:.2f}%')

        # 3) NCC (paper Eq. 5 -- train centres, test evaluation)
        ncc_ret, ncc_for = ncc_eval(model, full_train_loader, test_set, forget_classes)
        print(f'  NCC      retain={ncc_ret:.2f}%  forget={ncc_for:.2f}%')

        eval_rows.append({
            'seed': seed, 'ratio': ratio,
            'forget_classes': str(forget_classes),
            'out_retain':  round(out_ret,  2),  'out_forget':  round(out_for,  2),
            'lp_retain':   round(lp_ret,   2),  'lp_forget':   round(lp_for,   2),
            'ncc_retain':  round(ncc_ret,  2),  'ncc_forget':  round(ncc_for,  2),
        })

df_eval = pd.DataFrame(eval_rows)
print('\n=== Theta_o Evaluation (all seeds x ratios) ===')
print(df_eval.to_string(index=False))

# save alongside the config
eval_path = f'{CKPT_ROOT}/theta_o_eval.json'
with open(eval_path, 'w') as f:
    json.dump(eval_rows, f, indent=2)
print(f'\nSaved evaluation to {eval_path}')

## F. Export cmf_benchmark_config.json

In [ ]:
# A.6 fix: config contains the EXACT split protocol for all downstream notebooks.
config = {
    'repo_url':    'https://github.com/tiensinh2/CMF_Unlearning',
    'repo_commit': REPO_COMMIT,
    'dataset':     DATASET,
    'arch':        ARCH,
    'num_classes': NUM_CLASSES,
    'seeds':       SEEDS,
    'ratios':      RATIOS,
    'test_mode':   TEST_MODE,
    'ckpt_root':   CKPT_ROOT,
    # Paper-faithful pretrain config (§A.4)
    'pretrain': {
        'batch_size':    BATCH_SIZE,
        'epochs':        EPOCHS,
        'patience':      PATIENCE,
        'lr_init':       LR_INIT,
        'weight_decay':  WEIGHT_DECAY,
        'momentum':      MOMENTUM,
        'warmup_epochs': WARMUP_EPOCHS,
        'min_lr':        MIN_LR,
        'lr_scheduler':  'cosine_with_warmup',
        'optimizer':     'SGD',
    },
    # Split protocol (A.6 — loaded by ALL downstream notebooks)
    'split_protocol': 'stratified_random_mix',
    'split_files': {
        f'ratio{r}_seed{s}': {
            'forget': f'splits/forget_indices_ratio{r}_seed{s}.json',
            'retain': f'splits/retain_indices_ratio{r}_seed{s}.json',
        }
        for r in RATIOS for s in SEEDS
    },
    # Theta_o checkpoints
    'theta_o_ckpts': {
        str(s): f'pre_train/theta_o_seed{s}.pt' for s in SEEDS
    },
    # Evaluation convention
    'forget_baseline': 'oracle_forget_acc',
    'eval_comment': (
        'Forget accuracy compared against oracle retrain forget acc, '
        'NOT against 0, because under random-mix the oracle itself does not reach 0%.'
    ),
    # A.1 CMF deviation disclosed
    'cmf_disclosed_deviation': (
        'recompute_cmf() L2-normalizes features before averaging. '
        'Paper Algorithm 1 uses raw z_theta(x). Intentional in our CMF variant.'
    ),
}

config_path = f'{CKPT_ROOT}/cmf_benchmark_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print('Saved:', config_path)

## G. Summary

In [ ]:
print('=== NB1 Complete ===')
print(f'Checkpoint root : {CKPT_ROOT}')
print(f'Theta_o checkpoints : {[f"theta_o_seed{s}.pt" for s in SEEDS]}')
print(f'Split files     : forget_indices_ratio{{30|10}}_seed{{0|1|2}}.json')
print(f'Config file     : cmf_benchmark_config.json')
print(f'Eval file       : theta_o_eval.json')
print()
print('=== Theta_o Training Results ===')
print(pd.DataFrame(train_results).to_string(index=False))
print()
print('=== Theta_o Evaluation --- Output / LP / NCC (paper Table 1 Original row) ===')
print(df_eval[['seed', 'ratio',
               'out_retain',  'out_forget',
               'lp_retain',   'lp_forget',
               'ncc_retain',  'ncc_forget']].to_string(index=False))
print()
print('Next: Publish /kaggle/working/checkpoints/ as Kaggle dataset, then run NB2.')